# FRC Vision Scout — GPU pipeline (free Colab)

**Setup (do this once, ~4 clicks):**
1. `Runtime` → `Change runtime type` → Hardware accelerator = **T4 GPU** → Save
2. `Runtime` → **Run all**
3. When the last cell finishes it auto-downloads the annotated video, and prints metrics.

This runs the match through **high-resolution YOLO11x detection + SAHI sliced inference + ByteTrack tracking on the GPU at full frame rate** — no training needed for this first pass. It fixes the two problems from the CPU demo: low recall (small/airborne balls) and low frame rate.

Paste any cell error back to Claude and it will fix the notebook.

In [ ]:
#@title 1. Install + GPU check
!pip -q install ultralytics sahi yt-dlp >/dev/null 2>&1
import torch, ultralytics
print('ultralytics', ultralytics.__version__)
print('CUDA available:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU — set Runtime>GPU')
assert torch.cuda.is_available(), 'Enable a GPU runtime: Runtime > Change runtime type > T4 GPU'

In [ ]:
#@title 2. Provide the match clip (UPLOAD — Colab can't download from YouTube)
# Colab's datacenter IP is bot-gated by YouTube ("confirm you're not a bot"), so we do NOT
# download here. Upload a clip you already have on your computer (Claude gives you a 720p one:
# data/match_720p.mp4 in the repo). ~40MB, takes a few seconds.
import os, cv2
if not (os.path.exists('match.mp4') and os.path.getsize('match.mp4')>0):
    from google.colab import files
    print('Pick your local clip (e.g. match_720p.mp4)...')
    up=files.upload()
    src=list(up.keys())[0]
    if src!='match.mp4': os.replace(src,'match.mp4')
c=cv2.VideoCapture('match.mp4'); n=int(c.get(7))
print('clip:', int(c.get(3)),'x',int(c.get(4)),'@',round(c.get(5),1),'fps', n,'frames'); c.release()
assert n>0, 'No usable clip — re-run and upload match_720p.mp4.'

In [ ]:
#@title 3. Tracked detection at full frame rate (YOLO11x + ByteTrack, GPU)
# High imgsz = small/airborne ball recall; ByteTrack = persistent IDs; runs every frame on GPU.
from ultralytics import YOLO
model=YOLO('yolo11x.pt')
res=model.track(source='match.mp4', imgsz=1920, conf=0.04, iou=0.5,
                classes=[32],            # sports ball (FUEL); add 0=person later for refs
                tracker='bytetrack.yaml', persist=True, stream=True, verbose=False)
import cv2, numpy as np
cap=cv2.VideoCapture('match.mp4'); fps=cap.get(5); W=int(cap.get(3)); H=int(cap.get(4)); cap.release()
vw=cv2.VideoWriter('tracked.mp4', cv2.VideoWriter_fourcc(*'mp4v'), fps, (W,H))
counts=[]
for i,r in enumerate(res):
    f=r.orig_img.copy(); n=0
    if r.boxes is not None and r.boxes.id is not None:
        for b,tid in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.id.cpu().numpy().astype(int)):
            x1,y1,x2,y2=b.astype(int); cx,cy=(x1+x2)//2,(y1+y2)//2
            cv2.circle(f,(cx,cy),max(3,(x2-x1)//2),(0,255,255),2); n+=1
    elif r.boxes is not None:
        for b in r.boxes.xyxy.cpu().numpy():
            x1,y1,x2,y2=b.astype(int); cv2.circle(f,((x1+x2)//2,(y1+y2)//2),max(3,(x2-x1)//2),(0,255,255),2); n+=1
    cv2.putText(f,f'FUEL tracked: {n}',(8,28),cv2.FONT_HERSHEY_SIMPLEX,0.8,(0,0,255),2)
    vw.write(f); counts.append(n)
vw.release()
!ffmpeg -hide_banner -loglevel error -y -i tracked.mp4 -c:v libx264 -pix_fmt yuv420p tracked_h264.mp4
print('frames:',len(counts),'| mean balls/frame:',round(float(np.mean(counts)),1),'| max:',int(np.max(counts)))

In [ ]:
#@title 4. (Max recall) SAHI sliced inference on a sample frame
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
import cv2
det=AutoDetectionModel.from_pretrained(model_type='ultralytics', model_path='yolo11x.pt',
                                       confidence_threshold=0.04, device='cuda:0')
cap=cv2.VideoCapture('match.mp4'); cap.set(1,int(cap.get(5)*90)); ok,frame=cap.read(); cap.release()
cv2.imwrite('frame.png',frame)
sp=get_sliced_prediction('frame.png', det, slice_height=384, slice_width=384,
                         overlap_height_ratio=0.25, overlap_width_ratio=0.25)
balls=[o for o in sp.object_prediction_list if o.category.id==32]
for o in balls:
    bb=o.bbox; cx,cy=int((bb.minx+bb.maxx)/2),int((bb.miny+bb.maxy)/2)
    cv2.circle(frame,(cx,cy),max(3,int((bb.maxx-bb.minx)/2)),(0,255,255),2)
cv2.putText(frame,f'SAHI FUEL: {len(balls)}',(8,28),cv2.FONT_HERSHEY_SIMPLEX,0.8,(0,0,255),2)
cv2.imwrite('sahi_frame.png',frame); print('SAHI balls on sample frame:',len(balls))

In [ ]:
#@title 5. Download results
from google.colab import files
print('Downloading tracked video + SAHI sample frame...')
files.download('tracked_h264.mp4')
files.download('sahi_frame.png')